# G1 and G3 Generation Demo

## Goal and scope
Demonstrate direct generation (G1) and human-analysis-mediated generation (G3) on the same approved reflection document. This notebook is the G1/G3 entry point; reusable generation, validation, and evaluation functions live in importable project modules. The separate G2 notebook consumes predicted analysis through the same request and output contracts.

## Conditions
- **G1: Direct baseline.** Provide the ordered reflection segments and rubric, with no segment analysis.
- **G2: Predicted-analysis intervention.** Provide the same reflection and rubric plus predicted scope and component-band labels. Develop and improve its inference pipeline separately.
- **G3: Human-analysis headroom.** Provide the same reflection and rubric plus human-annotated scope and component-band labels aligned to segment IDs.

Use the same generator model, shared prompt content, output schema, and generation settings across conditions. Keep reflection text, annotations, model outputs, and traces within approved protected storage.

## G1 and G3 demo

This notebook is a thin entry point over `reflection_assessment_feedback/`, the shared implementation intended for this experiment's other notebooks as well. The request interface already accepts G2 with predicted analysis; G2 does not need a separate output schema or generation runner.

The prepared split can contain alternative candidates for the same segment. The G3 loader therefore accepts only documents with one usable, non-review candidate per segment and rejects ambiguous documents rather than selecting or merging labels. Out-of-scope segments map to `N`; in-scope component targets `0-3` are interpreted using the expert rubric defined in `reflection_assessment_feedback/rubric.py`.

Generation is deliberately opt-in. Reflection text and model output are protected research data. Use only an approved document and provider; enable remote LangSmith tracing only if the project has approved sending the full prompt and output to that tracing workspace.

### 1. Select the test essay

Choose `ESSAY_NAME` from `R1`–`R4` and `ESSAY_RECORD_INDEX` from the G3-eligible documents for that essay. Several test documents can share one essay name, so the index selects among those documents in sorted `document_id` order. The cell shows choice counts and segment count, not IDs or protected text.

In [9]:
from pathlib import Path

import pandas as pd

from reflection_assessment_feedback import (
    EXPERT_RUBRIC,
    eligible_document_ids,
    load_human_annotated_document,
)

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
DATA_PATH = PROJECT_ROOT / "data/provisional_segment_splits/provisional_test_modeling_wide.csv"
FEEDBACK_PATH = (
    PROJECT_ROOT.parent
    / "prebi_ap4_v1/rl1_reflection_analytics/data/raw/human_feedback/testset_human_feedback_by_essay.csv"
)
if not FEEDBACK_PATH.is_file():
    raise FileNotFoundError("The protected testset feedback copy was not found.")

feedback_rows = pd.read_csv(FEEDBACK_PATH, dtype={"document_id": str})
required_feedback_columns = {"essay_name", "document_id", "full_text", "feedback_text"}
if not required_feedback_columns.issubset(feedback_rows.columns):
    raise ValueError("The testset feedback copy is missing required columns.")
feedback_rows["essay_name"] = feedback_rows["essay_name"].astype(str).str.strip().str.upper()
feedback_rows["document_id"] = feedback_rows["document_id"].astype(str).str.strip()
if feedback_rows["document_id"].duplicated().any():
    raise ValueError("The testset feedback copy must have one row per document.")
if set(feedback_rows["essay_name"]) - {"R1", "R2", "R3", "R4"}:
    raise ValueError("Unexpected essay_name; expected only R1, R2, R3, or R4.")

eligible_ids = set(eligible_document_ids(DATA_PATH))
ESSAY_NAME = "R4"  # Choose R1, R2, R3, or R4.
ESSAY_RECORD_INDEX = 0  # Zero-based index among G3-eligible records for the selected essay.
essay_choices = (
    feedback_rows.loc[
        (feedback_rows["essay_name"] == ESSAY_NAME)
        & feedback_rows["document_id"].isin(eligible_ids)
    ]
    .sort_values("document_id")
    .reset_index(drop=True)
)
if essay_choices.empty:
    raise ValueError(f"No G3-eligible test documents are available for {ESSAY_NAME}.")
if not 0 <= ESSAY_RECORD_INDEX < len(essay_choices):
    raise IndexError(f"ESSAY_RECORD_INDEX must be from 0 to {len(essay_choices) - 1} for {ESSAY_NAME}.")

human_feedback_record = essay_choices.iloc[ESSAY_RECORD_INDEX]
document, human_analysis = load_human_annotated_document(
    DATA_PATH, human_feedback_record["document_id"]
)
if not str(human_feedback_record["full_text"]).strip() or not str(human_feedback_record["feedback_text"]).strip():
    raise ValueError("The selected test document is missing its full text or human feedback.")
available_choice_counts = {
    essay_name: int(
        feedback_rows.loc[
            (feedback_rows["essay_name"] == essay_name)
            & feedback_rows["document_id"].isin(eligible_ids)
        ].shape[0]
    )
    for essay_name in ("R1", "R2", "R3", "R4")
}
print(
    f"Selected {ESSAY_NAME} choice {ESSAY_RECORD_INDEX}; "
    f"eligible choices by essay: {available_choice_counts}; segments: {len(document.segments)}"
)

Selected R4 choice 0; eligible choices by essay: {'R1': 1, 'R2': 4, 'R3': 5, 'R4': 4}; segments: 37


### 2. Inspect the human reference

The next cell displays the selected original full reflection and the segment-level human scope/band annotations. These structured scores are distinct from the free-text human feedback. The comparison view below brings both references together with G1/G3 outputs. Notebook output contains protected research data; do not share or commit the notebook with these outputs.

In [10]:
analysis_by_segment = {item.segment_id: item for item in human_analysis.segments}

print("ORIGINAL FULL REFLECTION")
print(str(human_feedback_record["full_text"]))

print("\nG3 HUMAN ANNOTATION SCORES (segment-level; 0=absent, 1-3=rubric bands, N=out of scope)")
for segment in document.segments:
    annotation = analysis_by_segment[segment.segment_id]
    labels = ", ".join(
        f"{dimension}={annotation.component_bands[dimension]}"
        for dimension in ("SW", "UA", "HA")
    )
    print(f"Segment {segment.order}: scope={annotation.scope}; {labels}")

print("\nFREE-TEXT HUMAN FEEDBACK")
print(str(human_feedback_record["feedback_text"]))

ORIGINAL FULL REFLECTION
emgi02
1
Reflexion Vignette Gelelektrophorese
2
1. Kognitive Aktivierung
3
Meiner Meinung nach erfolgt eine kognitive Aktivierung, wenn die Studierenden „ins Grübeln“ kommen und selbst auf eine Lösung kommen müssen, die noch nicht klar auf der Hand liegt. Eine kognitive Aktivierung erfolgt also, wenn das Vorwissen der Studierenden aktiviert und verknüpft wird. Dabei sollte die Aktivierung auf einem zu den Studierenden relativ hohen Niveau stattfinden, denn wenn die Aufgabe zu einfach ist und damit das Wissen bereits einfach abrufbar ist, bleibt eine Aktivierung aus. → anspruchsvolle Lernziele können nur erreicht werden, wenn auch anspruchsvolle Lernprozesse stattfinden. (vgl. Fauth 2018, S.2)
4
Die kognitive Aktivierung erfolgte in der Vignette, indem gefragt wurde, um was es sich bei der Pinken Farbe in der Skizze handelt. Die Frage wurde so gestellt und gleichzeitig auf das Gesuchte gezeigt, dass wahrscheinlich jeder Studierende die Aufgabenstellung an sich v

### 3. Set tracing policy

The provider client is created only in the explicit generation cell below, after the approval gate. LangSmith tracing remains disabled by default. To opt in, copy `.env.example` to `.env`, configure `LANGSMITH_API_KEY` and `LANGSMITH_PROJECT`, then set `ENABLE_LANGSMITH_TRACING = True`. The Python generation span and nested LangChain model call will be sent to that project. Traces can contain the full reflection, prompt, and model output; enable tracing only after confirming that the project and its data-retention policy are approved for this corpus.

In [11]:
ENABLE_LANGSMITH_TRACING = True

### 4. Run G1 and G3 (external model call)

Run only after confirming that this document may be processed by the configured provider. LangSmith tracing is disabled by default because it may capture full prompts and outputs. Enable it only with explicit approval for the tracing workspace and data retention policy.

In [12]:
import importlib
import os
from dotenv import dotenv_values, load_dotenv

langsmith_dotenv_path = PROJECT_ROOT / ".env"
langsmith_dotenv = dotenv_values(langsmith_dotenv_path)
load_dotenv(langsmith_dotenv_path, override=False)
conflicting_langsmith_settings = [
    name
    for name in ("LANGSMITH_API_KEY", "LANGSMITH_PROJECT", "LANGSMITH_ENDPOINT")
    if langsmith_dotenv.get(name)
    and os.getenv(name)
    and langsmith_dotenv[name] != os.getenv(name)
]
if conflicting_langsmith_settings:
    raise RuntimeError(
        "LangSmith settings differ between .env and the active kernel environment: "
        f"{', '.join(conflicting_langsmith_settings)}. The kernel environment takes "
        "precedence; update or unset the conflicting value and restart the kernel. "
        "Secret values are not displayed."
    )

from langchain_google_genai import ChatGoogleGenerativeAI
import reflection_assessment_feedback.generator as generation_module
from reflection_assessment_feedback import make_generation_request

generation_module = importlib.reload(generation_module)
RubricGenerationRunner = generation_module.RubricGenerationRunner

APPROVE_EXTERNAL_PROCESSING = True
REPETITIONS = 1
assert APPROVE_EXTERNAL_PROCESSING, "Confirm provider processing is approved before running."
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved environment."
)
if ENABLE_LANGSMITH_TRACING:
    assert os.getenv("LANGSMITH_API_KEY"), (
        "Configure LANGSMITH_API_KEY in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_PROJECT"), (
        "Configure LANGSMITH_PROJECT in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
        "https://eu.api.smith.langchain.com"
    ), "Tracing is restricted to the LangSmith EU endpoint."

chat_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)
runner = RubricGenerationRunner(
    chat_model,
    enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
)
g1_request = make_generation_request(
    condition="G1",
    document=document,
    rubric=EXPERT_RUBRIC,
)
g3_request = make_generation_request(
    condition="G3",
    document=document,
    rubric=EXPERT_RUBRIC,
    analysis=human_analysis,
)

runs = {
    condition: [runner.generate(request, repetition=index + 1) for index in range(REPETITIONS)]
    for condition, request in (("G1", g1_request), ("G3", g3_request))
}
print({condition: len(condition_runs) for condition, condition_runs in runs.items()})
if ENABLE_LANGSMITH_TRACING:
    print(
        "LangSmith tracing targeted EU endpoint "
        f"for project {os.environ['LANGSMITH_PROJECT']}; check warnings above for upload failures."
    )

/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'G1': 1, 'G3': 1}
LangSmith tracing targeted EU endpoint for project prebi_v1; check warnings above for upload failures.


In [13]:
runs


{'G1': [GenerationRun(run_id='2bfd1c66-44ef-4edc-ba16-66ad5bff5641', condition='G1', repetition=1, model_name='gemini-3.5-flash-lite', generation_parameters={'temperature': 0.0}, prompt_version='1.3.0', output=GenerationOutput(assessment=Assessment(component_assessments=[DimensionAssessment(dimension_id='SW', band=2.2, justification='Die Verfasserin wählt eine konkrete Interaktionssituation aus der Vignette aus (Fragestellung zur pinken Farbe in der Skizze und anschließende Reaktion des Präsentierenden) und beschreibt diese nachvollziehbar. Es werden jedoch kaum tiefere persönliche Werte oder Core Reflections für die Wahrnehmung benannt, weshalb Band 3 knapp nicht erreicht wird.', evidence_segment_ids=['segment-829bb01c3fe571c41b6b', 'segment-c16a465f722e05bbdd25', 'segment-2ed98c1528e7d233e732']), DimensionAssessment(dimension_id='UA', band=2.1, justification='Die Analyse beleuchtet Ursachen für das Ausbleiben sofortiger Antworten und zieht fachdidaktische Literatur (Fauth 2018) sowie

### 5. Save paired demo runs

Save generated G1/G3 repetitions together as one comparison artifact under `PREBI_DATA_ROOT/g1-g3/demo-runs/`. Each save creates a new file and does not overwrite prior runs. Artifacts are matched back to the selected essay by document ID. For human evidence review, each assessment stores zero-based `evidence_segment_orders` alongside the original `evidence_segment_ids`, aligned by position; the orders are derived from the document, not generated by the model.

In [ ]:
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from reflection_assessment_feedback.generator import output_for_review

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before saving demo runs.")

protected_root = Path(protected_root_value).expanduser()
if not protected_root.is_absolute():
    raise ValueError("PREBI_DATA_ROOT must be an absolute path.")
protected_root = protected_root.resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")

run_directory = protected_root / "g1-g3" / "demo-runs"
run_directory.mkdir(parents=True, exist_ok=True, mode=0o700)
if not run_directory.resolve().is_relative_to(protected_root):
    raise ValueError("Demo-run storage must resolve within PREBI_DATA_ROOT.")

if set(runs) != {"G1", "G3"}:
    raise ValueError("Expected generated runs for both G1 and G3.")
if any(len(condition_runs) != REPETITIONS for condition_runs in runs.values()):
    raise ValueError("Each condition must contain exactly REPETITIONS generated runs.")

runs_by_repetition = {
    condition: {run.repetition: run for run in condition_runs}
    for condition, condition_runs in runs.items()
}
if any(len(repetition_runs) != REPETITIONS for repetition_runs in runs_by_repetition.values()):
    raise ValueError("Run repetitions must be unique within each condition.")
if set(runs_by_repetition["G1"]) != set(runs_by_repetition["G3"]):
    raise ValueError("G1 and G3 runs must have matching repetition numbers.")


def serialize_run(run):
    return {
        "run_id": run.run_id,
        "condition": run.condition,
        "repetition": run.repetition,
        "model_name": run.model_name,
        "generation_parameters": run.generation_parameters,
        "prompt_version": run.prompt_version,
        "output": output_for_review(run.output, document),
        "feedback_letter": run.feedback_letter,
    }


comparison_package = {
    "schema_version": "1.1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "document_id": document.document_id,
    "segment_count": len(document.segments),
    "rubric_id": EXPERT_RUBRIC.rubric_id,
    "rubric_version": EXPERT_RUBRIC.version,
    "pairs": [
        {
            "repetition": repetition,
            "G1": serialize_run(runs_by_repetition["G1"][repetition]),
            "G3": serialize_run(runs_by_repetition["G3"][repetition]),
        }
        for repetition in sorted(runs_by_repetition["G1"])
    ],
}
output_path = run_directory / f"comparison-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex}.json"
temporary_path = None
try:
    with tempfile.NamedTemporaryFile(
        mode="w", encoding="utf-8", dir=run_directory,
        prefix=".comparison-", suffix=".tmp", delete=False,
    ) as temporary_file:
        temporary_path = Path(temporary_file.name)
        json.dump(comparison_package, temporary_file, ensure_ascii=False, indent=2)
        temporary_file.write("\n")
    os.replace(temporary_path, output_path)
finally:
    if temporary_path is not None:
        temporary_path.unlink(missing_ok=True)

print(f"Saved {len(comparison_package['pairs'])} paired G1/G3 repetitions to protected storage.")

Saved 1 paired G1/G3 repetitions to protected storage.


In [ ]:
saved_comparison = json.loads(output_path.read_text(encoding="utf-8"))
assert saved_comparison["document_id"] == document.document_id
assert saved_comparison["schema_version"] == "1.1"
assert len(saved_comparison["pairs"]) == REPETITIONS
orders_by_id = {segment.segment_id: segment.order for segment in document.segments}
for pair in saved_comparison["pairs"]:
    assert pair["G1"]["condition"] == "G1"
    assert pair["G3"]["condition"] == "G3"
    assert pair["G1"]["repetition"] == pair["G3"]["repetition"] == pair["repetition"]
    for condition in ("G1", "G3"):
        for assessment in pair[condition]["output"]["assessment"]["component_assessments"]:
            assert assessment["evidence_segment_orders"] == [
                orders_by_id[segment_id] for segment_id in assessment["evidence_segment_ids"]
            ]
print("Saved comparison artifact passed JSON, pairing, and evidence-order checks.")

Saved comparison artifact passed JSON and pairing checks.


### 6. Compare human reference, G1, and G3

The view joins the selected R1–R4 feedback record and human segment scores with the newest saved G1/G3 comparison artifact for that same document. If there is no saved artifact for the chosen essay, run the approved generation cell and save its paired runs first. The rendered text is protected research data.

In [ ]:
import html
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env", override=False)
protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT before loading saved comparison runs.")
protected_root = Path(protected_root_value).expanduser().resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must resolve outside the project source tree.")
comparison_directory = protected_root / "g1-g3" / "demo-runs"
if not comparison_directory.resolve().is_relative_to(protected_root):
    raise ValueError("Comparison artifacts must be stored within PREBI_DATA_ROOT.")

matching_comparisons = []
for artifact_path in sorted(
    comparison_directory.glob("comparison-*.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
    if artifact.get("document_id") == document.document_id:
        matching_comparisons.append((artifact_path, artifact))
        break
if not matching_comparisons:
    raise FileNotFoundError(
        f"No saved G1/G3 runs for {ESSAY_NAME} choice {ESSAY_RECORD_INDEX}. "
        "Run the approved generation cell and save the paired runs first."
    )
comparison_path, comparison_package = matching_comparisons[0]
if not comparison_package.get("pairs"):
    raise ValueError("The selected comparison artifact has no paired runs.")

segment_orders = {segment.segment_id: segment.order for segment in document.segments}
annotations_by_id = {item.segment_id: item for item in human_analysis.segments}


def escaped(value):
    return html.escape(str(value), quote=True)


def annotation_score_rows():
    rows = []
    for segment in document.segments:
        annotation = annotations_by_id[segment.segment_id]
        labels = annotation.component_bands
        rows.append(
            "<tr>"
            f"<td>{segment.order}</td>"
            f"<td>{escaped(annotation.scope)}</td>"
            f"<td>{escaped(labels['SW'])}</td>"
            f"<td>{escaped(labels['UA'])}</td>"
            f"<td>{escaped(labels['HA'])}</td>"
            "</tr>"
        )
    return "".join(rows)


def assessment_rows(run_data):
    assessments = run_data["output"]["assessment"]["component_assessments"]
    rows = []
    for assessment in assessments:
        evidence = ", ".join(
            f"{segment_orders[segment_id]} ({segment_id})"
            for segment_id in assessment["evidence_segment_ids"]
        )
        rows.append(
            "<tr>"
            f"<td>{escaped(assessment['dimension_id'])}</td>"
            f"<td>{escaped(assessment['band'])}</td>"
            f"<td>{escaped(assessment['justification'])}</td>"
            f"<td>{escaped(evidence)}</td>"
            "</tr>"
        )
    return "".join(rows)


def condition_panel(condition, run_data):
    return (
        f"<section class='condition'><h3>{condition} assessment</h3>"
        "<div class='table-wrap'><table><thead><tr>"
        "<th>Dimension</th><th>Band</th><th>Justification</th><th>Evidence: order (ID)</th>"
        "</tr></thead><tbody>"
        f"{assessment_rows(run_data)}</tbody></table></div>"
        "<h4>Final letter</h4>"
        f"<pre>{escaped(run_data['feedback_letter'])}</pre></section>"
    )


pair_sections = []
for pair in comparison_package["pairs"]:
    if pair["G1"]["condition"] != "G1" or pair["G3"]["condition"] != "G3":
        raise ValueError("Saved comparison contains mismatched conditions.")
    if pair["G1"]["repetition"] != pair["G3"]["repetition"]:
        raise ValueError("Saved G1/G3 runs are not paired by repetition.")
    pair_sections.append(
        f"<h2>Repetition {escaped(pair['repetition'])}</h2>"
        "<div class='conditions'>"
        f"{condition_panel('G1', pair['G1'])}"
        f"{condition_panel('G3', pair['G3'])}"
        "</div>"
    )

comparison_html = f"""
<style>
.prebi-view {{ color: #17212b; font: 14px/1.5 -apple-system, BlinkMacSystemFont, sans-serif; }}
.prebi-view h2 {{ margin: 1.3em 0 .5em; }}
.prebi-view h3 {{ margin: .2em 0 .7em; }}
.prebi-view h4 {{ margin: 1em 0 .3em; }}
.prebi-view pre {{ white-space: pre-wrap; overflow-wrap: anywhere; font: inherit; margin: .4em 0 1em; }}
.prebi-view .reference {{ border-top: 2px solid #357266; padding-top: .8em; }}
.prebi-view .conditions {{ display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 1.2em; }}
.prebi-view .condition {{ min-width: 0; border-top: 1px solid #c6d0d5; padding-top: .6em; }}
.prebi-view .table-wrap {{ overflow-x: auto; }}
.prebi-view table {{ width: 100%; border-collapse: collapse; margin: .4em 0 1em; }}
.prebi-view th, .prebi-view td {{ text-align: left; vertical-align: top; border-bottom: 1px solid #d9e0e3; padding: .4em; }}
.prebi-view th {{ background: #edf3f1; }}
@media (max-width: 800px) {{ .prebi-view .conditions {{ grid-template-columns: 1fr; }} }}
</style>
<div class='prebi-view'>
  <h1>{escaped(ESSAY_NAME)} comparison</h1>
  <p>G3-eligible choice {ESSAY_RECORD_INDEX} · {len(document.segments)} segments · saved run file {escaped(comparison_path.name)}</p>
  <section class='reference'>
    <h2>Human reference</h2>
    <details open><summary>Original full reflection</summary><pre>{escaped(human_feedback_record['full_text'])}</pre></details>
    <h3>Human annotation scores by segment</h3>
    <p>Order is zero-based. 0 = absent; 1–3 = rubric bands; N = out of scope.</p>
    <div class='table-wrap'><table><thead><tr><th>Order</th><th>Scope</th><th>SW</th><th>UA</th><th>HA</th></tr></thead>
    <tbody>{annotation_score_rows()}</tbody></table></div>
    <h3>Free-text human feedback</h3>
    <pre>{escaped(human_feedback_record['feedback_text'])}</pre>
  </section>
  {''.join(pair_sections)}
</div>
"""
display(HTML(comparison_html))

Segment,Scope,SW,UA,HA
1,in_scope,0,2,0
2,in_scope,0,1,0
3,in_scope,0,2,0
4,in_scope,0,3,0
5,in_scope,0,1,0
6,in_scope,1,0,0
7,in_scope,0,1,0
8,in_scope,0,1,0
9,in_scope,1,0,0
10,in_scope,0,1,0
